# Phutball curriculum v4: the weights carry over (expert iteration, Qwen3.5-9B thinking on, board + men list)

Same scheduler, validation set and gate as v3 (curriculum_pp_colab.ipynb, llm_train/curriculum_pp.py), one change:
**each round continues the best adapter** (`sft_stop --init-adapter`) instead of training a fresh LoRA on the base model.

Why: in v3 every round trained a new LoRA from the bare base model on ~544 examples (its ~272 new + equal replay), so no
round built on the previous one and even the self-stop warm start was not carried over; the training set did not
accumulate either (150 puzzles a round, unlike STaR / ReST-EM, which retrain from base on a re-sampled WHOLE dataset).
Validation finished accuracy: v0 0.149, r1 0.320, then 0.30-0.37 for rounds 2-9: almost all the gain came in round 1.

v4:
- **round 1 forks v3's screen**: the same 1,600 puzzles x 8 samples (v3's r1/ outcomes and examples are copied, no
  re-sampling; the screen is re-run only to rebuild the scheduler state, and must give v3's ids), but the training
  CONTINUES the warm start adapter;
- **later rounds** continue the current best adapter; a rejected round's adapter is discarded, so the gate still
  guards against drift (more important now: training on its own samples round after round can compound a bias such as
  the NO WIN collapse);
- **replay 0.5x, in proportion to the bank** (~68% win-finding examples): the weights carry what earlier rounds taught,
  replay only guards against forgetting. Rounds 2-5 used replay balanced over the subtasks (--balance-replay): win-finding
  fell to ~53% of the training set and, with the weights carried over, rounds 3-5 slid into saying NO WIN on real wins
  (forced wins z -5.8, -8.3, -4.6; all rejected by the gate); from round 6 replay is proportional;
- **learning rate 1e-4** (v3: 5e-5; LoRA's best rate is ~10x full fine-tuning's).

v3 vs v4 then share round 1's data and differ only in whether the weights carry over. The warm start's evals (`v0`,
`start`) and v2's reference are copied from v3. **Runtime:** A100, High-RAM; ~1.2 h per round (round 1: training +
evals only, ~1 h). Resumable: re-run all cells after a disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request
OLD = '/content/drive/MyDrive/phutball/curriculum'            # v2 run
V3 = '/content/drive/MyDrive/phutball/curriculum_pp'                   # v3: round 1's screen and the start evals come from here
OUT = '/content/drive/MyDrive/phutball/curriculum_v4'; os.makedirs(OUT, exist_ok=True)
INIT = '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final'   # the warm start: v2's starting model too
POOL = 'llm_train/puzzles/pool_v4.jsonl.gz';     # pool_v2 minus the one-rule prevent puzzles, plus 140 regenerated ones
STATE = f'{OUT}/state.json'; BANK = f'{OUT}/bank.jsonl'; FMT = 'ascii+men'
TEST = 'llm_bench/data/bench_v2.jsonl'          # the benchmark with the regenerated prevent puzzles
VAL = 'llm_train/puzzles/val_v3.jsonl'; VAL_IDS = 'llm_train/puzzles/val_v3_ids.json'   # llm_train.make_val (in the repo)
CAP = 6144; SCREEN = 1600; N = 150; K = 8; ROUNDS = 15   # round 1 ~4.5 h, then ~1 h per round; raise ROUNDS any time (resumes)
LR = 1e-4; REPLAY = 0.5
PG = 25; EK = 4; FULL_EVERY = 5                   # test (benchmark): 25 per group x 4 samples, on round 1 and every 5th
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
def evaluate(name, model, samples=EK, bench=TEST, pg=PG):
    """bench: the test set (default) or VAL (gate: all 1,100 items, pg=0)."""
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); return
    !python -m llm_bench.run --bench {bench} --model {model} --name {name} --prompt-format {FMT} --thinking true --qwen-thinking-sampling --samples {samples} --max-tokens {CAP} --per-group {pg} --concurrency 96 | tail -14
    !python -m llm_bench.run --bench {bench} --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --per-group {pg} --concurrency 48 | tail -14
    !cp llm_bench/results/{name}.jsonl llm_bench/results/{name}.summary.json llm_bench/results/{name}-forced.summary.json {EV}/
def best_adapter():
    return (json.load(open(STATE)).get('best') if os.path.exists(STATE) else None) or INIT

## 1. Starting point, from v3

The warm start's validation (`v0`, the gate's first 'best') and test (`start`) evals and v2's reference (`ref-*`) are the same models as in v3: copied, not re-run. Round 1's screen (ids, outcomes, examples) is copied from v3's r1/.


In [ ]:
import shutil
for f in glob.glob(f'{V3}/eval/v0*') + glob.glob(f'{V3}/eval/start*') + glob.glob(f'{V3}/eval/ref-*'):
    if not os.path.exists(f'{EV}/{os.path.basename(f)}'): shutil.copy(f, EV)
d = f'{OUT}/r1'; os.makedirs(d, exist_ok=True)
if not os.path.exists(STATE):
    !python -m llm_train.curriculum_pp screen --state {STATE} --pool {POOL} --exclude {VAL_IDS} --n {SCREEN} --out {d}/ids.json | head -3
    assert json.load(open(f'{d}/ids.json')) == json.load(open(f'{V3}/r1/ids.json')), "screen differs from v3's: stop"
for f in ('outcomes.jsonl', 'sft.jsonl', 'stats.json'):
    if not os.path.exists(f'{d}/{f}') and os.path.exists(f'{V3}/r1/{f}'): shutil.copy(f'{V3}/r1/{f}', d)
print(sorted(os.listdir(EV))[:8], '...'); print(sorted(os.listdir(d)))


## 2. Curriculum rounds (unattended)

In [ ]:
for R in range(1, ROUNDS + 1):
    d = f'{OUT}/r{R}'; os.makedirs(d, exist_ok=True)
    if not os.path.exists(f'{d}/ids.json'):
        if R == 1:
            !python -m llm_train.curriculum_pp screen --state {STATE} --pool {POOL} --exclude {VAL_IDS} --n {SCREEN} --out {d}/ids.json | head -16
        else:
            !python -m llm_train.curriculum_pp allocate --state {STATE} --pool {POOL} --exclude {VAL_IDS} --n {N} --round {R} --out {d}/ids.json | head -40
    if not os.path.exists(f'{d}/outcomes.jsonl'):                  # round 1: copied from v3 above
        print(f'=== round {R}: sampling from', best_adapter())
        !python -m llm_train.expert_iter --arm iid --adapter {best_adapter()} --pool {POOL} --ids {d}/ids.json --out {d} --k {K} --max-think {CAP} --seed {R} --prompt-format {FMT} --max-neg-ratio 1.0 2>&1 | grep --line-buffered -v -E "it/s\]|Processed prompts|Adding requests|INFO|WARNING"
    if not os.path.exists(f'{d}/train.jsonl'):
        !python -m llm_train.curriculum_pp update --state {STATE} --round {R} --outcomes {d}/outcomes.jsonl --sft {d}/sft.jsonl --bank {BANK} --train-out {d}/train.jsonl --replay {REPLAY} | tail -40
    if not os.path.exists(f'{d}/adapter/final/adapter_model.safetensors'):
        print(f'=== round {R}: training continues', best_adapter())
        !python -m llm_train.sft_stop --data {d}/train.jsonl --out {d}/adapter --max-neg-ratio 1.0 --init-adapter {best_adapter()} --lr {LR} 2>&1 | grep --line-buffered -E "examples|continuing|mismatch|'loss'|saved|rror|Traceback"
    full = (R == 1 or R % FULL_EVERY == 0)
    if not os.path.exists(f'{EV}/v{R}-forced.summary.json') or (full and not os.path.exists(f'{EV}/r{R}-forced.summary.json')):
        server = serve({f'r{R}': f'{d}/adapter/final'})
        evaluate(f'v{R}', f'r{R}', samples=1, bench=VAL, pg=0)     # gate: validation set
        if full: evaluate(f'r{R}', f'r{R}')                         # test set (curves only)
        !pkill -f "vllm serve"
    if not os.path.exists(f'{d}/gate.txt'):
        !python -m llm_train.curriculum_pp gate --state {STATE} --round {R} --new {EV}/v{R} --adapter {d}/adapter/final --init-best {EV}/v0 --init-adapter {INIT} | tee {d}/gate.txt
print('curriculum done; best:', best_adapter())


## 3. Results

In [ ]:
names = ['start', 'ref-r9'] + [f'r{R}' for R in range(1, ROUNDS + 1)]   # test evals: warm start, v2's best, round 1 and every FULL_EVERY
keys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
        'win: near-miss negative', 'forced placement', 'block placement', 'prevent placement')
vkeys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
         'win: backward-trap, 2-jump', 'win: backward-trap, 3-jump', 'win: backward-trap, 4-jump', 'win: near-miss negative',
         'forced placement', 'block placement', 'prevent placement')
vnames = ['v0', 'ref-v9'] + [f'v{R}' for R in range(1, ROUNDS + 1)]
for label, suffix in (('VALIDATION, FINISHED', '.summary.json'), ('VALIDATION, BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in vnames if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\n{label}: pass@1, 100 per bucket")
    print(f"{'':28s}" + ''.join(f"{n:>7s}" for n in S))
    for k in vkeys:
        print(f"{k:28s}" + ''.join(f"{S[n][k]['accuracy']:>7.0%}" if k in S[n] else f"{'-':>7s}" for n in S))
    print(f"{'cut off (wins)':28s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>7.0%}" for n in S))
for label, suffix in (('FINISHED ON ITS OWN', '.summary.json'), ('BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in names if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\nTEST {label}: pass@1 / pass@{EK}  (ascii+men, thinking on, sampled, {CAP} tokens)")
    print(f"{'':26s}" + ''.join(f"{n:>14s}" for n in S))
    for k in keys:
        print(f"{k:26s}" + ''.join(f"{S[n][k]['accuracy']:>8.0%} /{S[n][k].get(f'pass@{EK}', float('nan')):>4.0%}" if k in S[n] else f"{'-':>14s}" for n in S))
    print(f"{'cut off (wins)':26s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>14.0%}" for n in S))
    print(f"{'mean tokens':26s}" + ''.join(f"{S[n].get('mean_completion_tokens', 0):>14.0f}" for n in S))
v3 = {n: json.load(open(f'{V3}/eval/{n}.summary.json')) for n in vnames if os.path.exists(f'{V3}/eval/{n}.summary.json')}
print('\nv3 (from base) vs v4 (carried over): validation finished accuracy, wins + no-win + placements')
fin = lambda s: sum(s[k]['accuracy'] * s[k]['n'] for k in vkeys if k in s and k != 'prevent placement') / sum(s[k]['n'] for k in vkeys if k in s and k != 'prevent placement')
for n in vnames:
    a, b = (fin(v3[n]) if n in v3 else None), (fin(json.load(open(f'{EV}/{n}.summary.json'))) if os.path.exists(f'{EV}/{n}.summary.json') else None)
    if a is not None or b is not None: print(f"  {n:7s} v3 {a if a is not None else float('nan'):.3f}   v4 {b if b is not None else float('nan'):.3f}")
if os.path.exists(STATE):
    st = json.load(open(STATE))
    print('\nscheduler log:')
    for e in st['log']:
        if e['event'] == 'gate': print(f"  round {e['round']}: {'PROMOTED' if e['promoted'] else 'kept best'} ({e['reason']})")
        elif e['event'] == 'update': print(f"  round {e['round']}: finished rates {e['finished_rates']} | {e['moves']} | {e['status']}")
        elif e['event'] == 'train_set': print(f"  round {e['round']}: train {e['new']} new + {e['replay']} replay ({e['dropped_mastered']} mastered dropped)")

In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()